In [31]:
!pip install rdkit imbalanced-learn xgboost

import pandas as pd
import numpy as np

df=pd.read_csv('Mpro_bioactivity_v2.csv')
print("Total:",len(df))
print("Active:",sum(df['label']==1))
print("Inactive:",sum(df['label']==0))
print(df.head())

Total: 4946
Active: 195
Inactive: 4751
       chembl_id                                      smiles    IC50  pIC50  \
0  CHEMBL4000000      O=C(CC(C)CC(=O)c1ccccc1)NC1CC(=O)NC1=O  148.79  6.827   
1  CHEMBL4000001   O=C(CC(C)CC(=O)c1ccc(F)cc1)NC1CC(=O)NC1=O   89.53  7.048   
2  CHEMBL4000002  O=C(CC(C)CC(=O)c1ccc(Cl)cc1)NC1CC(=O)NC1=O  167.89  6.775   
3  CHEMBL4000003  O=C(CC(C)CC(=O)c1ccc(Br)cc1)NC1CC(=O)NC1=O  338.19  6.471   
4  CHEMBL4000004  O=C(CC(C)CC(=O)c1ccc(OC)cc1)NC1CC(=O)NC1=O   82.92  7.081   

   label  
0      1  
1      1  
2      1  
3      1  
4      1  


In [33]:
from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np

def smiles_to_fingerprint(smiles):
  mol = Chem.MolFromSmiles(smiles)
  if mol is None:
    return None
  fp = AllChem.GetMorganFingerprintAsBitVect(
      mol, radius=2, nBits=2048
  )
  return list(fp)

print("Generating fingerprints...")
df['fingerprint'] = df['smiles'].apply(smiles_to_fingerprint)
df = df.dropna(subset=['fingerprint'])

X = np.array(df['fingerprint'].tolist())
y = np.array(df['label'])

print("Done!")
print("X shape:", X.shape)
print("Active:", sum(y == 1))
print("Inactive:", sum(y == 0))

Generating fingerprints...
Done!
X shape: (4946, 2048)
Active: 195
Inactive: 4751


In [34]:
from sklearn.model_selection import train_test_split,StratifiedKFold,cross_val_score
from imblearn.over_sampling import SMOTE

X_train,X_test,y_train,y_test=train_test_split(
  X,y,
  test_size=0.2,
  random_state=42,
  stratify=y
)
print("Train set:",X_train.shape)
print("Test set:",X_test.shape)
smote=SMOTE(random_state=42)
X_train_smote,y_train_smote=smote.fit_resample(
    X_train,y_train
)
print("\nAfter SMOTE:")
print("Active:",sum(y_train_smote==1))
print("Inactive:",sum(y_train_smote==0))
print("Total:",len(y_train_smote))




Train set: (3956, 2048)
Test set: (990, 2048)

After SMOTE:
Active: 3800
Inactive: 3800
Total: 7600


In [35]:
from sklearn.ensemble import RandomForestClassifier
rf_model=RandomForestClassifier(
    n_estimators=100,
    random_state=42,
)
skf=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
print("Training Random Forest...")
cv_scores_rf=cross_val_score(
    rf_model,
    X_train_smote,
    y_train_smote,
    cv=skf,
    scoring='roc_auc',
)
print("CV AUC scores:",cv_scores_rf.round(3))
print("Mean AUC:",cv_scores_rf.mean().round(3))
print("\Random Forest done!")

<>:17: SyntaxWarning: invalid escape sequence '\R'
<>:17: SyntaxWarning: invalid escape sequence '\R'
/tmp/ipykernel_1608/1509345625.py:17: SyntaxWarning: invalid escape sequence '\R'
  print("\Random Forest done!")


Training Random Forest...
CV AUC scores: [1. 1. 1. 1. 1.]
Mean AUC: 1.0
\Random Forest done!


In [37]:
from xgboost import XGBClassifier
xgb_model=XGBClassifier(
    n_estimators=100,
    random_state=42,
    eval_metric='logloss',
    verbosity=0,
)
print("Training XGBoost...")
cv_scores_xgb=cross_val_score(
    xgb_model,
    X_train_smote,
    y_train_smote,
    cv=skf,
    scoring='roc_auc',

)
print("CV AUC scores:",cv_scores_xgb.round(3))
print("Mean AUC:",cv_scores_xgb.mean().round(3))
print("\nXGBoost done!")

Training XGBoost...
CV AUC scores: [1. 1. 1. 1. 1.]
Mean AUC: 1.0

XGBoost done!


In [40]:
from sklearn.metrics import (roc_auc_score,f1_score,
  matthews_corrcoef,classification_report,confusion_matrix)
print("="*40)
print("RANDOM FOREST-TEST SET RESULTS")
print("="*40)

# Fit the RandomForest model on the SMOTE-resampled training data
rf_model.fit(X_train_smote, y_train_smote)

rf_pred=rf_model.predict(X_test)
rf_pred_proba=rf_model.predict_proba(X_test)[:,1]
print("AUC-ROC:",np.round(roc_auc_score(y_test,rf_pred_proba),3))
print("F1-score:",np.round(f1_score(y_test,rf_pred),3))
print("MCC:",np.round(matthews_corrcoef(y_test,rf_pred),3))
print("\nClassification report:")
print(classification_report(y_test,rf_pred))


RANDOM FOREST-TEST SET RESULTS
AUC-ROC: 0.999
F1-score: 0.94
MCC: 0.939

Classification report:
              precision    recall  f1-score   support

           0       1.00      0.99      1.00       951
           1       0.89      1.00      0.94        39

    accuracy                           0.99       990
   macro avg       0.94      1.00      0.97       990
weighted avg       1.00      0.99      1.00       990



In [41]:
import joblib
joblib.dump(rf_model,'rf_model.pkl')
joblib.dump(xgb_model,'xgb_model.pkl')
print("Models saved successfully!")


Models saved successfully!


In [42]:
from google.colab import files
files.download('rf_model.pkl')
files.download('xgb_model.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [43]:
files.download('Mpro_bioactivity_v2.csv')
#

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>